# Analyse exploratoire — Iris Dataset
EDA préalable à la recherche automatisée de modèles ML.

## 1. Setup & imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 100

## 2. Chargement des données

In [ ]:
df = pd.read_csv("../data/Iris.csv")
df.drop(columns=["Id"], inplace=True)
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Qualité des données

In [ ]:
print("Valeurs manquantes :")
print(df.isnull().sum())

print(f"\nDoublons : {df.duplicated().sum()}")

print("\nDistribution des classes :")
print(df["Species"].value_counts())

## 4. Analyse exploratoire (EDA)

### 4.1 Distribution des features

In [ ]:
features = ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feat in zip(axes.flat, features):
    for species, grp in df.groupby("Species"):
        sns.kdeplot(grp[feat], ax=ax, label=species, fill=True, alpha=0.3)
    ax.set_title(feat)
    ax.legend(fontsize=8)
plt.suptitle("Distribution des features par espèce", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

### 4.2 Heatmap des corrélations

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(
    df[features].corr(),
    annot=True, fmt=".2f",
    cmap="coolwarm", vmin=-1, vmax=1,
    square=True, linewidths=0.5
)
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.show()

### 4.3 Pairplot

In [ ]:
sns.pairplot(df, hue="Species", diag_kind="kde", plot_kws={"alpha": 0.6})
plt.suptitle("Pairplot — séparabilité des espèces", y=1.01, fontsize=13)
plt.show()

### 4.4 Boxplots par espèce

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feat in zip(axes.flat, features):
    sns.boxplot(data=df, x="Species", y=feat, ax=ax)
    ax.set_title(feat)
    ax.set_xlabel("")
plt.suptitle("Boxplots par espèce", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 5. Tableau récapitulatif EDA

In [ ]:
summary = (
    df.groupby("Species")[features]
    .agg(["mean", "std"])
    .round(2)
)
summary.columns = [" ".join(col) for col in summary.columns]
summary

## 6. Conclusion

**Features les plus discriminantes :**
- `PetalLengthCm` et `PetalWidthCm` séparent très bien les 3 espèces (distributions quasi non-chevauchantes entre setosa et les deux autres, légère ambiguïté entre versicolor et virginica).
- `SepalLengthCm` offre une séparation partielle.
- `SepalWidthCm` est la feature la moins discriminante (distributions très chevauchantes).

**Corrélations :**
- Forte corrélation positive entre les deux features Petal (~0.96) — potentiel de réduction de dimensionnalité.
- Corrélation modérée entre SepalLength et les features Petal.

**Qualité des données :**
- Aucune valeur manquante, dataset équilibré (50 samples par classe).
- 3 doublons potentiels à surveiller (vérifier si intentionnels).

**Implications pour le ML :**
- Tâche de classification multi-classes (3 classes) bien séparable — on s'attend à un F1 macro élevé (> 0.95) avec la plupart des classifieurs.
- Les features Petal seront probablement les plus importantes dans les modèles arborescents.
- La séparation de setosa est triviale ; le vrai défi est versicolor vs virginica.